# JOINTS 2026 Main Notebook

Objective: Memprediksi penjualan tiket harian D4-D10 untuk setiap pasangan film dan klaster bioskop berdasarkan penjualan D1-D3 dengan menggunakan MASE sebagai metrik evaluasi.

# Import modules dan libraries

In [1]:
import os, glob, random, re
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import sdm
from sdm import TableTensor

# Import datasets

In [2]:
_DATA_DIR_CANDIDATES = [
    "../data", "data", "../../data",
    *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/**/", recursive=True),
]
DATA_DIR = next((d for d in _DATA_DIR_CANDIDATES if d and os.path.exists(os.path.join(d, "test_history.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("test_history.csv not found in any candidate DATA_DIR - set it manually.")
print("DATA_DIR =", os.path.abspath(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date_show"])
test_hist = pd.read_csv(f"{DATA_DIR}/test_history.csv", parse_dates=["date_show"])
test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["date_show"])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
holidays = pd.read_csv(f"{DATA_DIR}/holidays.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA_DIR}/ticket_prices.csv")

DATA_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\data


### External datasets
Calendar files built by external_add.ipynb in the external/ folder.

In [3]:
_EXTERNAL_DIR_CANDIDATES = ["../external", "external", "../../external", *glob.glob("/kaggle/input/*")]
EXTERNAL_DIR = next((d for d in _EXTERNAL_DIR_CANDIDATES if os.path.exists(os.path.join(d, "cuti_bersama.csv"))), None)
if EXTERNAL_DIR is None:
    raise FileNotFoundError("cuti_bersama.csv not found in any candidate EXTERNAL_DIR - set it manually.")
print("EXTERNAL_DIR =", os.path.abspath(EXTERNAL_DIR))

cuti = pd.read_csv(f"{EXTERNAL_DIR}/cuti_bersama.csv", parse_dates=["date"])
ramadan = pd.read_csv(f"{EXTERNAL_DIR}/ramadan.csv", parse_dates=["date"])
school = pd.read_csv(f"{EXTERNAL_DIR}/school_holidays.csv", parse_dates=["date"])
extra_holidays = pd.read_csv(f"{EXTERNAL_DIR}/extra_national_holidays.csv", parse_dates=["date"])

EXTERNAL_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\external


# Aggregate Dataset
Menggabungkan semua file menjadi satu dataset utuh dengan satu baris per film, bioskop, dan tanggal target (D4-D10), sama seperti format test.csv. Train dan test digabung dalam satu tabel dengan penanda is_train, dan semua kolom mentah dari holidays, harga tiket, dan metadata film ikut digabung tanpa diolah.

### Penentuan D1
D1 train adalah tanggal pertama film tayang di minimal 50% dari jumlah bioskop maksimalnya, agar hari preview tidak dihitung sebagai D1. D1 test adalah tanggal pertama film di test_history. Film train yang sudah tayang sejak awal data atau belum mencapai D10 sebelum data berakhir dibuang.

In [4]:
D1_MIN_SHARE = 0.5

cinemas_per_day = train.groupby(["movie_title", "date_show"])["cinema_ids"].nunique()
first_show = train.groupby("movie_title")["date_show"].min()
wide_release = cinemas_per_day[cinemas_per_day >= D1_MIN_SHARE * cinemas_per_day.groupby(level=0).transform("max")]
train_d1 = wide_release.reset_index().groupby("movie_title")["date_show"].min()
train_d1 = train_d1[
    (first_show[train_d1.index] > train["date_show"].min())
    & (train_d1 + pd.Timedelta(days=9) <= train["date_show"].max())
]
test_d1 = test_hist.groupby("movie_title")["date_show"].min()

print(f"film train: {len(train_d1)} | film test_history: {len(test_d1)} | film test: {test['movie_title'].nunique()}")

film train: 205 | film test_history: 163 | film test: 160


### Pasangan film-bioskop
Mengambil transaksi D1-D10 (train) atau D1-D3 (test) setiap pasangan film dan bioskop, lalu menjadikan setiap hari sebagai kolom dan mengisi hari tanpa transaksi dengan 0. Pasangan train hanya diambil jika masih ada penjualan di D3, mengikuti test yang hanya berisi pasangan yang masih tayang di D3.

In [5]:
DAILY_COLS = {"total_ticket": "t", "total_show": "show"}

def build_window(hist, d1_map, n_days):
    df = hist[hist["movie_title"].isin(d1_map.index)].copy()
    df["day"] = (df["date_show"] - df["movie_title"].map(d1_map)).dt.days + 1
    df = df[df["day"].between(1, n_days)]
    parts = []
    for col, prefix in DAILY_COLS.items():
        p = df.pivot_table(index=["movie_title", "cinema_ids"], columns="day", values=col, fill_value=0)
        p = p.reindex(columns=range(1, n_days + 1), fill_value=0)
        p.columns = [f"{prefix}{d}" for d in p.columns]
        parts.append(p)
    out = pd.concat(parts, axis=1).reset_index()
    out["d1_date"] = out["movie_title"].map(d1_map)
    return out

train_window = build_window(train, train_d1, 10)
test_window = build_window(test_hist, test_d1, 3)

train_pairs = train_window[train_window["t3"] > 0].reset_index(drop=True)
test_pairs = test_window.merge(test[["movie_title", "cinema_ids"]].drop_duplicates(), on=["movie_title", "cinema_ids"])
print(f"pasangan train: {len(train_pairs)} | pasangan test: {len(test_pairs)}")

pasangan train: 8007 | pasangan test: 10373


### Format harian
Mengubah setiap pasangan menjadi 7 baris D4-D10, sehingga satu baris mewakili satu tanggal yang diprediksi. Input yang dibawa hanya D1-D3 (tiket, okupansi, jumlah tayang), dan total_ticket adalah target yang kosong untuk test.

In [6]:
HORIZONS = range(4, 11)
INPUT_COLS = [f"{p}{d}" for p in DAILY_COLS.values() for d in (1, 2, 3)]

def to_long(pairs, with_target):
    rows = []
    for h in HORIZONS:
        r = pairs[["movie_title", "cinema_ids", "d1_date"] + INPUT_COLS].copy()
        r["horizon"] = h
        r["date_show"] = r["d1_date"] + pd.Timedelta(days=h - 1)
        r["total_ticket"] = pairs[f"t{h}"].values if with_target else np.nan
        rows.append(r)
    return pd.concat(rows, ignore_index=True)

train_long = to_long(train_pairs, with_target=True).assign(is_train=1)
test_long = to_long(test_pairs, with_target=False).assign(is_train=0).merge(
    test[["id", "movie_title", "cinema_ids", "date_show"]], on=["movie_title", "cinema_ids", "date_show"], how="left"
)
full = pd.concat([train_long, test_long], ignore_index=True)
print(f"baris train: {len(train_long)} | baris test: {len(test_long)} | baris test.csv: {len(test)}")

baris train: 56049 | baris test: 72611 | baris test.csv: 72611


### Penggabungan semua sumber data
Menambahkan kolom mentah dari sumber lain ke dataset:
- city_name: kota tempat bioskop berada.
- day_tipe, holiday_tipe, holiday_name: tipe hari dan status libur tanggal target dari holidays.
- price: harga tiket kota untuk tipe hari tanggal target dari ticket_prices.
- age_rating, genre, producer, director, writer, casts: metadata film dari movies, dicocokkan lewat judul tanpa akhiran format (3D, IMAX).

In [7]:
FORMAT_PATTERN = r"\s*\((?:IMAX )?[23]D\)\s*$"
CINEMA_CITY = pd.concat([train, test_hist]).drop_duplicates("cinema_ids").set_index("cinema_ids")["city_name"]
PRICE_DAY = {"weekday": "Weekday", "friday": "Friday", "weekend": "Weekend"}

def join_sources(df):
    n = len(df)
    df = df.copy()
    df["city_name"] = df["cinema_ids"].map(CINEMA_CITY)
    df = df.merge(holidays.rename(columns={"date": "date_show"}), on="date_show", how="left")
    df["price_day"] = df["day_tipe"].map(PRICE_DAY)
    df = df.merge(prices.rename(columns={"ceil": "price"}), on=["city_name", "price_day"], how="left")
    df["base_title"] = df["movie_title"].str.replace(FORMAT_PATTERN, "", regex=True)
    df = df.merge(movies.rename(columns={"original_title": "base_title"}), on="base_title", how="left")
    assert len(df) == n
    return df

full = join_sources(full)
print(f"full: {full.shape}")
full.isna().sum()[lambda s: s > 0]

full: (128660, 27)


total_ticket     72611
id               56049
holiday_name    122732
dtype: int64

### External datasets join
Adds the raw columns of the external calendar files on the target date, left empty on dates outside them:
- cuti_bersama_name: the holiday a collective leave day belongs to.
- school_break: the school break period (DKI Jakarta).

In [8]:
def join_external(df):
    n = len(df)
    df = df.merge(cuti[["date", "name"]].rename(columns={"date": "date_show", "name": "cuti_bersama_name"}), on="date_show", how="left")
    df = df.merge(school[["date", "period"]].rename(columns={"date": "date_show", "period": "school_break"}), on="date_show", how="left")
    assert len(df) == n
    return df

full = join_external(full)
print(f"full: {full.shape}")
full.groupby("is_train")[["cuti_bersama_name", "school_break"]].count()

full: (128660, 29)


,cuti_bersama_name,school_break
is_train,,
0,2392,12540
1,1141,6208


In [9]:
full.head()

,movie_title,cinema_ids,d1_date,t1,t2,t3,show1,show2,show3,horizon,...,price,base_title,age_rating,genre,producer,director,writer,casts,cuti_bersama_name,school_break
0,28 YEARS LATER,015555fbcd8faef6b55a854605df0a87,2025-06-18,261.0,210.0,137.0,18.0,17.0,12.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,2025-06-18,1376.0,1033.0,946.0,23.0,23.0,23.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
2,28 YEARS LATER,04a2de432a8901363a05137a8e8de1d2,2025-06-18,0.0,83.0,85.0,0.0,3.0,3.0,4,...,40000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
3,28 YEARS LATER,073b764928864ed8fa0b6b115bfdd685,2025-06-18,159.0,55.0,43.0,9.0,3.0,2.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
4,28 YEARS LATER,108f9a4d181f12fd1b7e3c9adae38478,2025-06-18,643.0,458.0,447.0,30.0,19.0,20.0,4,...,55000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN


# Preprocessing
Cleans the aggregated dataset: removes broken training rows, turns raw columns into model-ready values, drops columns without usable signal, and builds the target.

### Drop incomplete days
Days in train with far fewer rows than the 15-day median are recording gaps (7-16 June 2025), not real drops in demand. Training rows whose target date is a gap day get a fake zero, and rows whose D1-D3 touches a gap day get a wrong scale, so both are removed.

In [10]:
rows_per_day = train.groupby("date_show").size().reindex(
    pd.date_range(train["date_show"].min(), train["date_show"].max()), fill_value=0
)
row_share = rows_per_day / rows_per_day.rolling(15, center=True, min_periods=5).median()
GAP_DAYS = row_share.index[row_share < 0.7]

def drop_gap_rows(df):
    d13_on_gap = np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(GAP_DAYS) for k in range(3)], axis=0)
    broken = (df["is_train"] == 1) & (df["date_show"].isin(GAP_DAYS) | d13_on_gap)
    return df[~broken].reset_index(drop=True)

print("gap days:", [d.date().isoformat() for d in GAP_DAYS])
n_before = len(full)
full = drop_gap_rows(full)
print(f"dropped train rows: {n_before - len(full)} | full: {full.shape}")

gap days: ['2025-06-07', '2025-06-09', '2025-06-11', '2025-06-13', '2025-06-14', '2025-06-15', '2025-06-16']
dropped train rows: 1378 | full: (127282, 29)


### Calendar flags
Turns the raw calendar columns into 0/1 flags for the target date. Holiday names are not kept, because the test holidays never appear in train:
- is_holiday: national holiday.
- is_cuti_bersama: collective leave day.
- is_school_holiday: school break (DKI Jakarta).

In [11]:
def add_calendar_flags(df):
    df = df.copy()
    df["is_holiday"] = (df["holiday_tipe"] == "holiday").astype(int)
    df["is_cuti_bersama"] = df["cuti_bersama_name"].notna().astype(int)
    df["is_school_holiday"] = df["school_break"].notna().astype(int)
    return df

full = add_calendar_flags(full)
full.groupby("is_train")[["is_holiday", "is_cuti_bersama", "is_school_holiday"]].mean().round(3)

,is_holiday,is_cuti_bersama,is_school_holiday
is_train,,,
0,0.041,0.033,0.173
1,0.054,0.018,0.114


### Genre columns
A film can have several genres ("Action, War"), so genre becomes one 0/1 column per genre.

In [12]:
def add_genres(df, genre_cols=None):
    dummies = df["genre"].str.get_dummies(sep=", ").add_prefix("genre_")
    dummies.columns = dummies.columns.str.replace(" ", "_")
    if genre_cols is not None:
        dummies = dummies.reindex(columns=genre_cols, fill_value=0)
    return pd.concat([df, dummies], axis=1), list(dummies.columns)

full, GENRE_COLS = add_genres(full)
COMMON_GENRES = [g for g in GENRE_COLS if full.loc[full["is_train"] == 1, g].mean() >= 0.02]   # rarer genres did not help
print(f"genres: {len(GENRE_COLS)} | used as features: {len(COMMON_GENRES)}")
full.loc[full["is_train"] == 1, GENRE_COLS].mean().sort_values(ascending=False).round(3).head(10)

genres: 27 | used as features: 15


genre_Horror       0.375
genre_Drama        0.328
genre_Action       0.212
genre_Thriller     0.146
genre_Comedy       0.120
genre_Animation    0.070
genre_Mystery      0.063
genre_Fantasy      0.056
genre_Superhero    0.052
genre_Family       0.043
dtype: float64

### Category columns
Columns with a small fixed set of text values become pandas categories, which LightGBM reads directly. Train and test share one table, so they get the same categories.

In [13]:
CAT_COLS = ["city_name", "day_tipe", "age_rating"]
CAT_LEVELS = {col: full[col].astype("category").cat.categories for col in CAT_COLS}

def set_categories(df):
    df = df.copy()
    for col in CAT_COLS:
        df[col] = pd.Categorical(df[col], categories=CAT_LEVELS[col])
    return df

full = set_categories(full)
full[CAT_COLS].nunique()

city_name     69
day_tipe       3
age_rating     4
dtype: int64

### Drop columns without signal
Removes raw columns that are replaced by the flags and genre columns above, helper columns used only for the joins, and free-text credits (producer, director, writer, casts), since most names in test films never appear in train.

In [14]:
DROP_COLS = [
    "holiday_tipe", "holiday_name", "cuti_bersama_name", "school_break", "genre",
    "price_day", "base_title", "producer", "director", "writer", "casts",
]

def drop_unused(df):
    return df.drop(columns=DROP_COLS)

full = drop_unused(full)
print(f"full: {full.shape}")
print(list(full.columns))

full: (127282, 48)
['movie_title', 'cinema_ids', 'd1_date', 't1', 't2', 't3', 'show1', 'show2', 'show3', 'horizon', 'date_show', 'total_ticket', 'is_train', 'id', 'city_name', 'day_tipe', 'price', 'age_rating', 'is_holiday', 'is_cuti_bersama', 'is_school_holiday', 'genre_Action', 'genre_Adventure', 'genre_Animation', 'genre_Biography', 'genre_Comedy', 'genre_Crime', 'genre_Disaster', 'genre_Documentary', 'genre_Drama', 'genre_Family', 'genre_Fantasy', 'genre_Gothic', 'genre_History', 'genre_Horror', 'genre_Music', 'genre_Musical', 'genre_Mystery', 'genre_Religi', 'genre_Romance', 'genre_Sci-Fi', 'genre_Sport', 'genre_Superhero', 'genre_Survival', 'genre_Suspense', 'genre_Teen', 'genre_Thriller', 'genre_War']


### Target
The target is tickets divided by the pair's D1-D3 average (clipped at 1, the same scale the MASE metric uses), so a model with an L1 objective directly optimizes the competition metric.

In [15]:
def add_target(df):
    df = df.copy()
    df["scale"] = df[["t1", "t2", "t3"]].mean(axis=1).clip(lower=1)
    df["target_ratio"] = df["total_ticket"] / df["scale"]
    return df

full = add_target(full)
full.loc[full["is_train"] == 1].groupby("horizon")["target_ratio"].median().round(3)

horizon
4     0.915
5     0.616
6     0.467
7     0.360
8     0.163
9     0.000
10    0.000
Name: target_ratio, dtype: float64

# Feature Engineering

### Days-off blocks
Holidays differ mostly by how many days off they create in a row: a holiday on Thursday with a cuti bersama Friday makes a 4-day weekend, while a holiday on a Sunday adds almost nothing. A day counts as off if it is a weekend, a national holiday (holidays.csv plus the two holidays just before it from external/) or a cuti bersama day. The calendar starts 30 days before the first D1 so that blocks crossing the start of the data keep their full length:
- off_block_len: length of the block of consecutive days off that contains the target date (0 on a working day).

D1-D3 versions of this feature were tried and removed: in train, films opening on a long weekend drop hard afterwards, so the model predicted a steep drop for the Lebaran 2026 films, whose demand actually stays high through Eid week (LB 0.48785 vs 0.47566).

On 2026-10-09 D1-D3 days off came back as win_off / off_rel (cell "Days off in the D1-D3 window"): CV 0.3643 -> 0.3577 in the clean pipeline; with the Cinepoint columns the model sees the real Lebaran curve, so the drop above does not come back here (it still matters for a pipeline without them).

In [16]:
cal = pd.DataFrame({"date": pd.date_range(full["d1_date"].min() - pd.Timedelta(days=30), full["date_show"].max())})
NATIONAL_HOLIDAYS = pd.concat([holidays.loc[holidays["holiday_tipe"] == "holiday", "date"], extra_holidays["date"]])
cal["off"] = (
    (cal["date"].dt.dayofweek >= 5) | cal["date"].isin(NATIONAL_HOLIDAYS) | cal["date"].isin(cuti["date"])
).astype(int)
block_id = (cal["off"] != cal["off"].shift()).cumsum()
cal["block_len"] = cal.groupby(block_id)["off"].transform("size") * cal["off"]
BLOCK_LEN = cal.set_index("date")["block_len"]

def add_off_block(df):
    return df.assign(off_block_len=df["date_show"].map(BLOCK_LEN))

full = add_off_block(full)

print("longest blocks:")
print(cal[cal["block_len"] >= 4].groupby(block_id)["date"].agg(["min", "max", "size"]).reset_index(drop=True))
full.groupby("is_train")["off_block_len"].value_counts(normalize=True).unstack().round(3)

longest blocks:
         min        max  size
0 2025-03-28 2025-04-07    11
1 2025-05-10 2025-05-13     4
2 2025-05-29 2025-06-01     4
3 2025-06-06 2025-06-09     4
4 2025-12-25 2025-12-28     4
5 2026-02-14 2026-02-17     4
6 2026-03-18 2026-03-24     7


off_block_len,0,1,2,3,4,7,11
is_train,,,,,,,
0,0.658,0.005,0.228,0.019,0.047,0.042,NaN
1,0.657,0.006,0.213,0.072,0.051,NaN,0.001


### Pair features
Features from the pair's own D1-D3 sales:
- trend: D3 tickets compared with D1, to see whether sales rise or fall.
- t3_share: share of D3 in the D1-D3 total.
- show_mean: average number of shows over D1-D3.
- tickets_per_show3: tickets per show on D3.
- active_days: how many of D1-D3 had sales.

In [17]:
D13 = ["t1", "t2", "t3"]

def add_pair_features(df):
    df = df.copy()
    df["trend"] = (df["t3"] + 1) / (df["t1"] + 1)
    df["t3_share"] = df["t3"] / df[D13].sum(axis=1)
    df["show_mean"] = df[["show1", "show2", "show3"]].mean(axis=1)
    df["tickets_per_show3"] = df["t3"] / df["show3"].replace(0, np.nan)
    df["active_days"] = (df[D13] > 0).sum(axis=1)
    return df

full = add_pair_features(full)
full[["trend", "t3_share", "show_mean", "tickets_per_show3", "active_days"]].describe().round(2)

,trend,t3_share,show_mean,tickets_per_show3,active_days
count,127282.00,127282.00,127282.00,127282.00,127282.00
mean,11.13,0.33,8.34,32.42,2.90
std,51.19,0.16,11.10,31.17,0.36
min,0.01,0.00,0.33,0.25,1.00
25%,0.56,0.24,3.33,12.14,3.00
50%,0.89,0.32,5.00,24.00,3.00
75%,1.49,0.40,9.00,42.20,3.00
max,1199.00,1.00,216.00,421.33,3.00


### Film features
National D1-D3 sales of the film over all its cinemas, because a film rises and falls in a similar way everywhere, and the LB showed big and rising films hold up better in the test period. Train and test rows are computed from their own windows, since a few titles appear in both (previews in train):
- mv_t1, mv_t2, mv_t3: national tickets per day.
- mv_trend: national D3 compared with D1.
- mv_clusters: number of cinemas still selling on D3.
- pair_share: the pair's share of the film's national D1-D3 tickets.

In [18]:
def film_features(window):
    g = window.groupby("movie_title")
    out = g[D13].sum().add_prefix("mv_")
    out["mv_clusters"] = g["t3"].apply(lambda s: (s > 0).sum())
    out["mv_trend"] = (out["mv_t3"] + 1) / (out["mv_t1"] + 1)
    return out

FILM_COLS = ["mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend"]

def add_film_features(df, window):
    df = df.copy()
    feats = film_features(window)
    for col in FILM_COLS:
        df[col] = df["movie_title"].map(feats[col]).values
    df["pair_share"] = df[D13].sum(axis=1) / df[["mv_t1", "mv_t2", "mv_t3"]].sum(axis=1)
    return df

is_tr = full["is_train"] == 1
full = pd.concat([add_film_features(full[is_tr], train_window), add_film_features(full[~is_tr], test_window)]).sort_index()
full.groupby("is_train")[FILM_COLS + ["pair_share"]].median().round(3)

,mv_t1,mv_t2,mv_t3,mv_clusters,mv_trend,pair_share
is_train,,,,,,
0,11615.0,12300.0,12249.0,87.0,0.869,0.008
1,22042.0,18511.0,17704.0,81.0,0.923,0.010


### Cinema features
Average D1-D3 scale of each cinema over all films in train and test_history, as a measure of how big the cinema is that does not use any D4-D10 target:
- cl_scale: average pair scale of the cinema.
- cl_movies: number of films the cinema showed.

In [19]:
all_windows = pd.concat([train_window, test_window], ignore_index=True)
cinema_feats = (
    all_windows.assign(pair_scale=all_windows[D13].mean(axis=1))
    .groupby("cinema_ids")
    .agg(cl_scale=("pair_scale", "mean"), cl_movies=("movie_title", "nunique"))
)

def add_cinema_features(df):
    return df.join(cinema_feats, on="cinema_ids")

full = add_cinema_features(full)
cinema_feats.describe().round(2)

,cl_scale,cl_movies
count,121.00,121.00
mean,192.95,192.13
std,165.41,73.64
min,18.77,5.00
25%,106.63,145.00
50%,160.53,189.00
75%,221.98,239.00
max,1348.26,347.00


### Weekday and price features
- dow, d1_dow: day of week of the target date and of D1, since where the weekend falls inside D1-D10 shifts the curve.
- price_ratio: ticket price on the target date compared with the average price over D1-D3.

In [20]:
DAY_TIPE_BY_DATE = holidays.set_index("date")["day_tipe"]
PRICE_MAP = prices.set_index(["city_name", "price_day"])["ceil"]

def add_weekday_price(df):
    df = df.copy()
    df["dow"] = df["date_show"].dt.dayofweek
    df["d1_dow"] = df["d1_date"].dt.dayofweek

    def price_on(dates):
        keys = pd.MultiIndex.from_arrays([df["city_name"].astype(str), dates.map(DAY_TIPE_BY_DATE).map(PRICE_DAY)])
        return pd.Series(PRICE_MAP.reindex(keys).values, index=df.index)

    d13_price = pd.concat([price_on(df["d1_date"] + pd.Timedelta(days=k)) for k in range(3)], axis=1).mean(axis=1)
    df["price_ratio"] = df["price"] / d13_price
    return df

full = add_weekday_price(full)
full[["dow", "d1_dow", "price_ratio"]].describe().round(3)

,dow,d1_dow,price_ratio
count,127282.0,127282.000,127282.000
mean,3.0,2.772,0.977
std,2.0,0.726,0.144
min,0.0,0.000,0.615
25%,1.0,2.000,0.875
50%,3.0,3.000,0.955
75%,5.0,3.000,1.105
max,6.0,6.000,1.500


### Days off around the D1-D3 window and the target day
Which of D1, D2, D3 and the target day are days off (and whether a national holiday or a weekend), whether the days next to the target are off, and how many days off fall between D4 and the target.

In [21]:
def is_off(dates):
    return ((dates.dt.dayofweek >= 5) | dates.isin(NATIONAL_HOLIDAYS) | dates.isin(cuti["date"])).astype(int)

def add_window_offdays(df):
    """days off (weekend, national holiday, cuti bersama) in the D1-D3 window, on and around the target day, and between D4 and the target"""
    df = df.copy()
    day = lambda k: df["d1_date"] + pd.Timedelta(days=k)
    df["off_d3"] = is_off(day(2))
    df["win_off"] = sum(is_off(day(k)) for k in range(3))
    df["win_hol"] = sum(day(k).isin(NATIONAL_HOLIDAYS).astype(int) for k in range(3))
    df["win_wk"] = sum((day(k).dt.dayofweek >= 5).astype(int) for k in range(3))
    df["tgt_off"] = is_off(df["date_show"])
    df["off_rel"] = df["tgt_off"] - df["win_off"] / 3
    df["tgt_hol"] = df["date_show"].isin(NATIONAL_HOLIDAYS).astype(int)
    df["tgt_prev_off"] = is_off(df["date_show"] - pd.Timedelta(days=1))
    df["tgt_next_off"] = is_off(df["date_show"] + pd.Timedelta(days=1))
    df["between_off"] = sum(((df["horizon"] > h) & (is_off(day(h - 1)) == 1)).astype(int) for h in range(4, 11))
    df["between_off_share"] = df["between_off"] / (df["horizon"] - 4).clip(lower=1)
    return df

OFF_COLS = ["off_d3", "win_off", "win_hol", "win_wk", "tgt_off", "off_rel",
            "tgt_hol", "tgt_prev_off", "tgt_next_off", "between_off", "between_off_share"]
full = add_window_offdays(full)
full.groupby("is_train")[OFF_COLS].mean().round(3)

,off_d3,win_off,win_hol,win_wk,tgt_off,off_rel,tgt_hol,tgt_prev_off,tgt_next_off,between_off,between_off_share
is_train,,,,,,,,,,,
0,0.692,1.059,0.178,0.723,0.342,-0.011,0.041,0.353,0.342,1.200,0.425
1,0.749,1.098,0.241,0.838,0.343,-0.023,0.054,0.345,0.340,1.118,0.404


### Film profile
What kind of film it is, from external/film_profile.csv: local or foreign, sequel / franchise, film kind, source material, country of origin, and the admissions of the previous film in the franchise. Test films whose facts were only public after 2025-09-30 get NaN (7 of 163), and so does a predecessor count published after that date.

In [22]:
PROFILE_CUTOFF = pd.Timestamp("2025-09-30")
ORIGINS = ["Indonesia", "United States", "South Korea", "Japan", "Thailand"]   # any other country: code 5

def load_film_profile():
    p = pd.read_csv(f"{EXTERNAL_DIR}/film_profile.csv")
    late = (p["split"] == "test") & ((p["info_public_before_cutoff"] == "no") | (pd.to_datetime(p["fact_source_date"], errors="coerce") > PROFILE_CUTOFF))
    pred_known = pd.to_datetime(p["predecessor_published"], errors="coerce") <= PROFILE_CUTOFF
    first_origin = p["origin_country"].astype(str).str.split(",").str[0].str.strip()
    prof = pd.DataFrame({
        "movie_title": p["movie_title"],
        "is_local": p["is_local"].astype(float),
        "is_sequel": p["is_sequel_or_franchise"].astype(float),
        "origin_code": first_origin.map({k: i for i, k in enumerate(ORIGINS)}).fillna(len(ORIGINS)).astype(float),
        "pred_adm_log": np.where(pred_known, np.log1p(p["predecessor_admissions_id"]), np.nan),
    })
    prof.loc[late.values, prof.columns[1:]] = np.nan
    return prof.drop_duplicates("movie_title").set_index("movie_title")

FILM_PROFILE = load_film_profile()
PROFILE_COLS = list(FILM_PROFILE.columns) + ["local_horror"]

def add_film_profile(df):
    df = df.copy()
    for c in FILM_PROFILE.columns:
        df[c] = df["movie_title"].map(FILM_PROFILE[c])
    df["local_horror"] = df["is_local"] * df["genre_Horror"]
    return df

full = add_film_profile(full)
print("films without a profile:", full.loc[full["is_local"].isna()].groupby("is_train")["movie_title"].nunique().to_dict())
full.groupby("is_train")[PROFILE_COLS].mean().round(3)

films without a profile: {0: 7}


,is_local,is_sequel,origin_code,pred_adm_log,local_horror
is_train,,,,,
0,0.531,0.336,1.225,14.955,0.209
1,0.564,0.309,0.753,14.545,0.286


### Late-start pairs
first_active, eff_age, last_ratio, active_mean: cinemas that start the film after D1 (same as main_legit).

In [23]:
def add_late_start(df):
    df = df.copy()
    t = df[D13].values
    first = np.select([t[:, 0] > 0, t[:, 1] > 0, t[:, 2] > 0], [1, 2, 3], 4)
    last = np.select([t[:, 2] > 0, t[:, 1] > 0], [t[:, 2], t[:, 1]], t[:, 0])
    df["first_active"] = first
    df["eff_age"] = df["horizon"] - first + 1
    df["last_ratio"] = last / df["scale"]
    df["active_mean"] = t.sum(axis=1) / np.clip((t > 0).sum(axis=1), 1, None) / df["scale"]
    return df

full = add_late_start(full)
full.groupby("is_train")["first_active"].value_counts(normalize=True).unstack().round(3)

first_active,1,2,3
is_train,,,
0,0.916,0.066,0.018
1,0.928,0.055,0.018


### Competition from other films' openings
New films opening at the same cinema / nationally after our D1 and up to the target day, and the strength of the other films' opening days on the target date and on our D1-D3 dates; train from train pairs, test from test pairs (same as main_legit).

In [24]:
COMP_MIN_SIZE = 50   # film D1-D3 mean below this is too small for the date strength

def opening_pairs(df):
    return df.drop_duplicates(["movie_title", "cinema_ids"])[["movie_title", "cinema_ids", "d1_date", "t1", "t2", "t3", "scale"]]

def film_openings(df):
    f = opening_pairs(df).groupby(["movie_title", "d1_date"])[D13].sum().reset_index()
    f["m"] = f[D13].mean(axis=1)
    return f[f["m"] >= COMP_MIN_SIZE].reset_index(drop=True)

def typical_opening_shape(df):
    """usual film D_k / mean(D1-D3), k = 1..3, by release weekday"""
    f = film_openings(df)
    log_r = lambda g, k: np.average(np.log((g[f"t{k}"] + 1) / (g["m"] + 1)), weights=np.sqrt(g["m"]))
    overall = [np.exp(log_r(f, k)) for k in (1, 2, 3)]
    by_dow = {dw: [np.exp(log_r(g, k)) for k in (1, 2, 3)] for dw, g in f.groupby(f["d1_date"].dt.dayofweek) if len(g) >= 5}
    return {dw: by_dow.get(dw, overall) for dw in range(7)}

def opening_surprise(df, shape):
    """log of each film's D1-D3 days vs the usual shape for its release weekday, one row per film and date"""
    rows = []
    for r in film_openings(df).itertuples():
        t = shape[r.d1_date.dayofweek]
        for k in range(3):
            rows.append((r.movie_title, r.d1_date + pd.Timedelta(days=k), np.log((getattr(r, f"t{k + 1}") + 1) / (r.m + 1) / t[k]), np.sqrt(r.m)))
    s = pd.DataFrame(rows, columns=["movie_title", "date", "s", "w"])
    return s.assign(sw=s["s"] * s["w"])

def date_strength(dates, titles, surprise):
    """size-weighted mean surprise of the other films in their D1-D3 on each date (NaN when there are none)"""
    tot = surprise.groupby("date")[["sw", "w"]].sum().reindex(pd.DatetimeIndex(dates)).values
    own = surprise.set_index(["date", "movie_title"])[["sw", "w"]].reindex(pd.MultiIndex.from_arrays([pd.DatetimeIndex(dates), titles])).fillna(0).values
    w = tot[:, 1] - own[:, 1]
    ok = w > 1e-9
    return np.where(ok, (tot[:, 0] - own[:, 0]) / np.where(ok, w, 1), np.nan)

def add_competition(df, ref, shape):
    """ref: rows whose pairs are the other films' openings"""
    df = df.reset_index(drop=True)
    others = opening_pairs(ref)
    own_size = opening_pairs(df).groupby("movie_title")["scale"].sum()

    key = df[["movie_title", "cinema_ids", "d1_date", "date_show"]].reset_index()
    m = key.merge(others[["cinema_ids", "movie_title", "d1_date", "scale"]], on="cinema_ids", suffixes=("", "_o"))
    m = m[(m["movie_title_o"] != m["movie_title"]) & (m["d1_date_o"] > m["d1_date"]) & (m["d1_date_o"] <= m["date_show"])]
    fresh = m[(m["date_show"] - m["d1_date_o"]).dt.days <= 2].groupby("index")["scale"].sum()
    df["comp_size_cin"] = np.log1p(m.groupby("index")["scale"].sum().reindex(df.index, fill_value=0).values / df["scale"].values)
    df["comp_fresh_cin"] = np.log1p(fresh.reindex(df.index, fill_value=0).values / df["scale"].values)

    films = others.groupby("movie_title").agg(fd1=("d1_date", "min"), fsize=("scale", "sum")).rename_axis("movie_o").reset_index()
    u = df[["movie_title", "d1_date", "date_show"]].drop_duplicates()
    mm = u.merge(films, how="cross")
    mm = mm[(mm["movie_o"] != mm["movie_title"]) & (mm["fd1"] > mm["d1_date"]) & (mm["fd1"] <= mm["date_show"])]
    u = u.merge(mm.groupby(["movie_title", "d1_date", "date_show"]).agg(comp_size_nat=("fsize", "sum")).reset_index(),
                how="left", on=["movie_title", "d1_date", "date_show"])

    surprise = opening_surprise(ref, shape)
    u["date_idx"] = date_strength(u["date_show"], u["movie_title"].values, surprise)
    own_days = np.vstack([date_strength(u["d1_date"] + pd.Timedelta(days=k), u["movie_title"].values, surprise) for k in range(3)])
    n_days = (~np.isnan(own_days)).sum(axis=0)
    u["own_idx"] = np.where(n_days > 0, np.nansum(own_days, axis=0) / np.maximum(n_days, 1), np.nan)
    u["date_idx_rel"] = u["date_idx"] - u["own_idx"]

    df = df.merge(u, how="left", on=["movie_title", "d1_date", "date_show"])
    df["comp_size_nat"] = np.log1p(df["comp_size_nat"].fillna(0) / df["movie_title"].map(own_size))
    return df

COMP_COLS = ["comp_size_cin", "comp_fresh_cin", "comp_size_nat", "date_idx", "own_idx", "date_idx_rel"]
OPENING_SHAPE = typical_opening_shape(full[full["is_train"] == 1])
is_tr = full["is_train"] == 1
full = pd.concat([add_competition(full[is_tr], full[is_tr], OPENING_SHAPE),
                  add_competition(full[~is_tr], full[~is_tr], OPENING_SHAPE)], ignore_index=True)
full.groupby("is_train")[COMP_COLS].mean().round(3)

,comp_size_cin,comp_fresh_cin,comp_size_nat,date_idx,own_idx,date_idx_rel
is_train,,,,,,
0,1.301,0.898,1.438,0.005,-0.012,0.020
1,1.256,0.874,1.485,-0.000,0.004,-0.003


### Cinepoint features (trial / assumption)
Real national daily admissions of each film from cinepoint.com (daily top box office), scraped into `external/cinepoint/`. Train period (Apr-Sep 2025) is before the cutoff; the test period (Oct 2025 - Mar 2026) is after it, so models using these columns are a "trial" (see logs.md).

Per film, from its D1 date:
- `cp_log_m13`: log of the mean real admissions on D1-D3.
- `retK`: real admissions on day K divided by that mean (K = 4..10), empty when the film is not in the daily list that day.
- `cutK`: smallest admissions in the list on that date divided by the same mean. When the film is missing from the list, its real sales were below this.
- `cp_cov`: our D1-D3 tickets over the real D1-D3 admissions, how much of the national sales our data sees.
- `ret_h`, `cut_h`: `retK` / `cutK` at the row's own horizon. `cp_ret_mean`, `cp_ndays`: mean of the known `retK` and number of known days.
- `n_ret_prev`: real admissions on the day before the row's target day / the D1-D3 mean (is the film on a rising or falling run going into the target day).
- `n_fill_ratio`: real admissions per showtime on the target day / the same on D1-D3 (theatres cutting shows, or shows emptying).

Train and test use the same code, so the model sees the same kind of signal in both. Set `USE_CINEPOINT = False` to switch the columns off.

A few dates of the daily files carry another day's numbers (the running totals do not add up, found by an audit): `external/cinepoint/cinepoint_daily_fix_*.csv` hold the re-scraped rows, which replace those dates when the charts are loaded.

In [25]:
USE_CINEPOINT = True
CP_FORMAT = r"\s*\((?:IMAX[^)]*|3D|2D|4DX)\)\s*$"
CP_HORIZONS = list(range(4, 11))
CP_DAY_FEATURES = ["n_ret_prev", "n_fill_ratio"]
CP_FEATURES = (["cp_log_m13", "cp_cov", "cp_ndays", "ret_h", "cut_h", "cp_ret_mean"]
               + [f"ret{k}" for k in CP_HORIZONS] + [f"cut{k}" for k in CP_HORIZONS] + CP_DAY_FEATURES)
def replace_chart_dates(charts, fix_files):
    """a few dates of the daily Cinepoint files carry another day's numbers (a film's running total does not equal the day before plus that day's admissions); the re-scraped rows replace those dates"""
    files = [f for f in fix_files if os.path.exists(f)]
    if not files:
        print("WARNING: no cinepoint_daily_fix_*.csv found, the mislabelled chart dates are NOT corrected (upload the fix files next to the daily files)")
        return charts
    fix = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in files], ignore_index=True)
    fix = fix[(fix["date"] >= charts["date"].min()) & (fix["date"] <= charts["date"].max())]
    print(f"chart dates replaced by the re-scraped rows: {fix['date'].nunique()} ({len(files)} fix file(s))")
    return pd.concat([charts[~charts["date"].isin(fix["date"].unique())], fix], ignore_index=True)

cp_key = lambda s: re.sub(r"[^a-z0-9]", "", re.sub(CP_FORMAT, "", s, flags=re.I).lower())

cp_files = [f"{EXTERNAL_DIR}/cinepoint/{n}" for n in ("cinepoint_daily_top_train.csv", "cinepoint_daily_top.csv")]
cp = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in cp_files if os.path.exists(f)], ignore_index=True)
cp = replace_chart_dates(cp, [f"{EXTERNAL_DIR}/cinepoint/{n}" for n in ("cinepoint_daily_fix_pre_cutoff.csv", "cinepoint_daily_fix_post_cutoff.csv")])
cp["key"] = cp["title"].map(cp_key)
CP_CUT = cp.groupby("date")["daily_admission"].min()
CP_DAILY = {k: g.groupby("date")["daily_admission"].sum() for k, g in cp.groupby("key")}
CP_SERIES = {k: g.groupby("date").agg(adm=("daily_admission", "sum"), st=("showtimes", "sum")) for k, g in cp.groupby("key")}

def cp_film_features(rows):
    pairs = rows.drop_duplicates(["movie_title", "cinema_ids"])
    films = pairs.groupby("movie_title").agg(d1=("d1_date", "min"), d13=("t1", "sum"), t2=("t2", "sum"), t3=("t3", "sum"))
    films["d13"] += films["t2"] + films["t3"]
    out = []
    for title, f in films.iterrows():
        rec = {"movie_title": title}
        daily = CP_DAILY.get(cp_key(title))
        days = pd.date_range(f["d1"], periods=10)
        real = daily.reindex(days) if daily is not None else pd.Series(np.nan, index=days)
        if real.iloc[:3].notna().any():
            m13 = real.iloc[:3].mean()
            rec.update(cp_log_m13=np.log1p(m13), cp_cov=f["d13"] / (3 * m13), cp_ndays=int(real.iloc[3:].notna().sum()))
            for k in CP_HORIZONS:
                rec[f"ret{k}"] = real.iloc[k - 1] / m13
                rec[f"cut{k}"] = CP_CUT.get(days[k - 1], np.nan) / m13
        out.append(rec)
    return pd.DataFrame(out)

def cp_day_features(rows):
    """per film x horizon: n_ret_prev = real national admissions on the day before the target / D1-D3 mean; n_fill_ratio = real admissions per showtime on the target day / the same on D1-D3"""
    d1s = rows.groupby("movie_title")["d1_date"].min()
    out = []
    for title, d1 in d1s.items():
        series = CP_SERIES.get(cp_key(title))
        if series is None:
            continue
        x = series.reindex(pd.date_range(d1, periods=10))
        adm13, st13 = x["adm"].iloc[:3].mean(), x["st"].iloc[:3].mean()
        if not (adm13 > 0 and st13 > 0):
            continue
        for k in CP_HORIZONS:
            st_k = x["st"].iloc[k - 1]
            fill = x["adm"].iloc[k - 1] / st_k if st_k > 0 else np.nan
            out.append({"movie_title": title, "horizon": k, "n_ret_prev": x["adm"].iloc[k - 2] / adm13, "n_fill_ratio": fill / (adm13 / st13)})
    return pd.DataFrame(out, columns=["movie_title", "horizon"] + CP_DAY_FEATURES)

def add_cinepoint(df):
    """Cinepoint columns per film from its D1 (train and test rows are done apart: a few titles occur in both with different D1 dates)"""
    parts = []
    for flag in sorted(df["is_train"].unique()):
        rows = df[df["is_train"] == flag]
        rows = rows.merge(cp_film_features(rows), on="movie_title", how="left")
        parts.append(rows.merge(cp_day_features(rows), on=["movie_title", "horizon"], how="left"))
    df = pd.concat(parts, ignore_index=True)
    for col in CP_FEATURES:
        if col not in df:
            df[col] = np.nan
    idx = df["horizon"].to_numpy() - 4
    ret = df[[f"ret{k}" for k in CP_HORIZONS]].to_numpy(dtype=float)
    cut = df[[f"cut{k}" for k in CP_HORIZONS]].to_numpy(dtype=float)
    df["ret_h"] = ret[np.arange(len(df)), idx]
    df["cut_h"] = cut[np.arange(len(df)), idx]
    df["cp_ret_mean"] = np.nanmean(ret, axis=1)
    return df

if USE_CINEPOINT:
    full = add_cinepoint(full)
    print(f"Cinepoint rows: {len(cp)}, dates {cp['date'].min().date()} to {cp['date'].max().date()}")
    display(full.groupby("is_train")[["cp_log_m13", "cp_cov", "cp_ndays", "ret_h", "cut_h"] + CP_DAY_FEATURES].agg(lambda s: s.notna().mean()).round(3))

chart dates replaced by the re-scraped rows: 36 (2 fix file(s))
Cinepoint rows: 6077, dates 2025-04-01 to 2026-03-31


C:\Users\leouw\AppData\Local\Temp\ipykernel_10632\3478676586.py:81: RuntimeWarning: Mean of empty slice
  df["cp_ret_mean"] = np.nanmean(ret, axis=1)


,cp_log_m13,cp_cov,cp_ndays,ret_h,cut_h,n_ret_prev,n_fill_ratio
is_train,,,,,,,
0,1.000,1.000,1.000,0.900,1.000,0.931,0.900
1,0.995,0.995,0.995,0.854,0.995,0.894,0.854


### Extra training rows (shifted windows)
Every train film's D1 moved s days later (window D(1+s)-D(3+s), targets the 7 days after it), through the same steps, with age_offset = s (same as main_legit).

In [26]:
AUG_SHIFTS = [1, 2, 3, 5, 7, 10, 14]

def build_shifted_rows(shift):
    d1 = train_d1 + pd.Timedelta(days=shift)
    d1 = d1[d1 + pd.Timedelta(days=9) <= train["date_show"].max()]
    window = build_window(train, d1, 10)
    df = to_long(window[window["t3"] > 0].reset_index(drop=True), with_target=True).assign(is_train=1)
    df = drop_gap_rows(join_external(join_sources(df)))
    df, _ = add_genres(add_calendar_flags(df), GENRE_COLS)
    df = add_target(drop_unused(set_categories(df)))
    df = add_weekday_price(add_cinema_features(add_film_features(add_pair_features(add_off_block(df)), window)))
    df = add_film_profile(add_late_start(add_window_offdays(df)))
    df = add_competition(df, full[full["is_train"] == 1], OPENING_SHAPE)
    if USE_CINEPOINT:
        df = add_cinepoint(df)
    return df.assign(age_offset=shift)

full["age_offset"] = 0
AUG = pd.concat([build_shifted_rows(s) for s in AUG_SHIFTS], ignore_index=True)
print(f"extra rows: {len(AUG)} ({len(AUG) / (full['is_train'] == 1).sum():.1f}x the real train rows)")
AUG.groupby("age_offset")["target_ratio"].agg(["size", "mean"]).round(3)

C:\Users\leouw\AppData\Local\Temp\ipykernel_10632\3478676586.py:81: RuntimeWarning: Mean of empty slice
  df["cp_ret_mean"] = np.nanmean(ret, axis=1)
C:\Users\leouw\AppData\Local\Temp\ipykernel_10632\3478676586.py:81: RuntimeWarning: Mean of empty slice
  df["cp_ret_mean"] = np.nanmean(ret, axis=1)
C:\Users\leouw\AppData\Local\Temp\ipykernel_10632\3478676586.py:81: RuntimeWarning: Mean of empty slice
  df["cp_ret_mean"] = np.nanmean(ret, axis=1)
C:\Users\leouw\AppData\Local\Temp\ipykernel_10632\3478676586.py:81: RuntimeWarning: Mean of empty slice
  df["cp_ret_mean"] = np.nanmean(ret, axis=1)
C:\Users\leouw\AppData\Local\Temp\ipykernel_10632\3478676586.py:81: RuntimeWarning: Mean of empty slice
  df["cp_ret_mean"] = np.nanmean(ret, axis=1)
C:\Users\leouw\AppData\Local\Temp\ipykernel_10632\3478676586.py:81: RuntimeWarning: Mean of empty slice
  df["cp_ret_mean"] = np.nanmean(ret, axis=1)


extra rows: 254483 (4.7x the real train rows)


C:\Users\leouw\AppData\Local\Temp\ipykernel_10632\3478676586.py:81: RuntimeWarning: Mean of empty slice
  df["cp_ret_mean"] = np.nanmean(ret, axis=1)


,size,mean
age_offset,,
1,50854,0.464
2,50833,0.482
3,48573,0.521
5,33873,0.635
7,28516,0.591
10,26786,0.410
14,15048,0.591


In [27]:
full.head()

,movie_title,cinema_ids,d1_date,t1,t2,t3,show1,show2,show3,horizon,...,ret9,cut9,ret10,cut10,n_ret_prev,n_fill_ratio,ret_h,cut_h,cp_ret_mean,age_offset
0,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,2025-12-05,58.0,97.0,59.0,7.0,5.0,3.0,4,...,NaN,0.081199,NaN,0.093355,0.897083,0.191792,0.297569,0.20081,0.274716,0
1,"13 DAYS, 13 NIGHTS",0191582d75f94a412c0fa12d36bcee3f,2025-12-05,135.0,191.0,148.0,7.0,5.0,5.0,4,...,NaN,0.081199,NaN,0.093355,0.897083,0.191792,0.297569,0.20081,0.274716,0
2,"13 DAYS, 13 NIGHTS",14b63ff2b104903ead7d884f25382e57,2025-12-05,7.0,22.0,3.0,2.0,1.0,1.0,4,...,NaN,0.081199,NaN,0.093355,0.897083,0.191792,0.297569,0.20081,0.274716,0
3,"13 DAYS, 13 NIGHTS",15c7dc37db27dd76f38cc419107f7c48,2025-12-05,0.0,50.0,23.0,0.0,3.0,2.0,4,...,NaN,0.081199,NaN,0.093355,0.897083,0.191792,0.297569,0.20081,0.274716,0
4,"13 DAYS, 13 NIGHTS",17e1e7da761f72bbb4e039ba08cc0ba0,2025-12-05,0.0,44.0,15.0,0.0,7.0,2.0,4,...,NaN,0.081199,NaN,0.093355,0.897083,0.191792,0.297569,0.20081,0.274716,0


# Regressor Models

### Setup and validation
All models predict target_ratio (tickets / D1-D3 average), so the mean absolute error on it is exactly the competition MASE. Validation is GroupKFold by film, so a validation film is never seen in training, like the test films. This CV is reliable for model and parameter choices, but not for anything tied to the test season, which only the LB can judge. Every model is then retrained on 100% of train to predict test.
- RUN_MODELS: which model cells actually train; the others are skipped.
- SEED: one constant for every source of randomness, so the results can be reproduced.
- MODEL_SEEDS: the tree models are trained once per seed and the predictions averaged with equal weights (never a seed picked by its CV score).
- ZERO_SNAP: tree models predict a small positive number where the true median is 0 (31.8% of train targets are exactly 0), so predictions below the threshold are set to 0. Out of fold this improved LightGBM from 0.3727 to 0.3673 and CatBoost from 0.3704 to 0.3653, also when the threshold is chosen on four folds and tested on the fifth.
- USE_EXTRA_ROWS, EXTRA_WEIGHT: the shifted-window rows of the fitting films are added in every fold and for the test fit (weight 0.5 in the tree models); validation is on real rows only.

Train rows that touch an incomplete day of train.csv (2025-06-01 to 06-16: far fewer shows or cinemas, a few days with no rows) are dropped, also from the extra shifted-window rows: their target and their D1-D3 base are under-reported, and the model is better without them (CV on the clean rows 0.2844 -> 0.2833).

`USE_ZERO_PROBABILITY`: 32% of the targets are exactly 0 (a pair that sold nothing that day), which L1 trees handle poorly. A classifier (smooth loss) predicts P(ratio == 0) and P(ratio < 0.2) and these two numbers become inputs of both models, computed out of fold by film for the train and extra rows and fitted on all train rows for the test rows. In a nested test (no leakage) the LGBM CV went from 0.2826 to 0.2789. The CV printed below is a little optimistic because the helper inputs of a training row come from classifiers that saw the validation fold's films.

In [28]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

RUN_MODELS = ["lgbm", "kumo", "nn"]   # ["lgbm"] alone is much faster
TARGET_POWER_LGBM = 0.75   # LGBM predicts tickets / national_ratio ** power, then multiplies it back (0 = the plain target)
TARGET_POWER_KUMO = 0.75   # same for Kumo (0 = the plain target)
RUN_CV = False   # False: skip the 5 validation folds (no CV scores, no out-of-fold predictions), only fit on all train rows and predict the test rows
MODEL_SEEDS = [42, 7, 123]
ZERO_SNAP = {"lgbm": 0.2, "catboost": 0.2, "xgboost": 0.2, "nn": 0.2}
N_FOLDS = 5
USE_EXTRA_ROWS = True   # shifted-window rows of the fitting films are added in every fold (validation stays on real rows)
EXTRA_WEIGHT = 0.5      # sample weight of the shifted-window rows in the tree models

NUM_FEATURES = [
    "t1", "t2", "t3", "show1", "show2", "show3", "horizon", "scale", "price",
    "trend", "t3_share", "show_mean", "tickets_per_show3", "active_days",
    "mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend", "pair_share", "cl_scale", "cl_movies",
    "dow", "d1_dow", "price_ratio",
    "is_holiday", "is_cuti_bersama", "is_school_holiday", "off_block_len",
] + OFF_COLS + [
    "first_active", "eff_age", "last_ratio", "active_mean",
] + COMP_COLS + ["age_offset"] + COMMON_GENRES + PROFILE_COLS + (CP_FEATURES if USE_CINEPOINT else [])
FEATURES = NUM_FEATURES + CAT_COLS

def incomplete_days(raw):
    """days on which train.csv is partial (found by a rule): far fewer shows or cinemas than the days around it (21-day median), or no rows at all"""
    g = raw.groupby("date_show").agg(shows=("total_show", "sum"), cinemas=("cinema_ids", "nunique")).reindex(pd.date_range(raw["date_show"].min(), raw["date_show"].max()))
    rel_shows = g["shows"] / g["shows"].rolling(21, center=True, min_periods=7).median()
    rel_cinemas = g["cinemas"] / g["cinemas"].rolling(21, center=True, min_periods=7).median()
    return pd.DatetimeIndex(g.index[(rel_shows < 0.75) | (rel_cinemas < 0.85) | g["shows"].isna()])

INCOMPLETE_DAYS = incomplete_days(train)

def touches_incomplete(df):
    """True for rows whose target day or any of D1-D3 is an incomplete day: their target and their D1-D3 base are under-reported"""
    return (df["date_show"].isin(INCOMPLETE_DAYS) | np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(INCOMPLETE_DAYS) for k in range(3)], axis=0)).values

drop_rows = (full["is_train"] == 1) & touches_incomplete(full)
print(f"incomplete train days: {len(INCOMPLETE_DAYS)} ({INCOMPLETE_DAYS.min().date()} .. {INCOMPLETE_DAYS.max().date()}); train rows dropped: {int(drop_rows.sum())} of {int((full['is_train'] == 1).sum())}")
train_df = full[(full["is_train"] == 1) & ~drop_rows].reset_index(drop=True)
test_df = full[full["is_train"] == 0].reset_index(drop=True)
CINEMA_CODE = {c: k for k, c in enumerate(sorted(pd.concat([train_df["cinema_ids"], test_df["cinema_ids"]]).unique()))}   # for the neural network
for df_ in (train_df, test_df):
    df_["cinema_code"] = df_["cinema_ids"].map(CINEMA_CODE)
X, y, groups = train_df[FEATURES + ["cinema_code"]], train_df["target_ratio"].values, train_df["movie_title"]
X_test = test_df[FEATURES + ["cinema_code"]]
AUG = AUG[~touches_incomplete(AUG)].reset_index(drop=True)   # same rule for the shifted-window rows
AUG["cinema_code"] = AUG["cinema_ids"].map(CINEMA_CODE)
X_aug, y_aug = AUG[FEATURES + ["cinema_code"]], AUG["target_ratio"].values
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups))

USE_ZERO_PROBABILITY = False   # helper inputs p_zero / p_low for both models (32% of the targets are exactly 0; a classifier with a smooth loss tells the L1 models where)
CLF_PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=2.0, verbose=-1)
if USE_ZERO_PROBABILITY:
    helper_targets = {"p_zero": y == 0, "p_low": y < 0.2}
    for col, target in helper_targets.items():
        train_df[col], AUG[col] = np.nan, np.nan
        for tr_idx, va_idx in FOLDS:   # out of fold by film: a row never sees a classifier trained on its own film
            clf = LGBMClassifier(random_state=SEED, **CLF_PARAMS).fit(X[FEATURES].iloc[tr_idx], target[tr_idx])
            train_df.loc[va_idx, col] = clf.predict_proba(X[FEATURES].iloc[va_idx])[:, 1]
            in_fold = AUG["movie_title"].isin(set(groups.iloc[va_idx])).values
            AUG.loc[in_fold, col] = clf.predict_proba(X_aug[FEATURES][in_fold])[:, 1]
        test_df[col] = LGBMClassifier(random_state=SEED, **CLF_PARAMS).fit(X[FEATURES], target).predict_proba(X_test[FEATURES])[:, 1]
    NUM_FEATURES = NUM_FEATURES + list(helper_targets)
    FEATURES = NUM_FEATURES + CAT_COLS
    X, X_test, X_aug = train_df[FEATURES + ["cinema_code"]], test_df[FEATURES + ["cinema_code"]], AUG[FEATURES + ["cinema_code"]]
    print(f"helper inputs added: mean p_zero train {train_df['p_zero'].mean():.3f} (real share {(y == 0).mean():.3f}) test {test_df['p_zero'].mean():.3f}")

OOF, TEST_PRED, SCORES = {}, {}, {}

def mase_ratio(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def fit_rows(idx):
    """real rows idx + the shifted-window rows of the same films; is_extra marks the shifted rows"""
    X_fit, y_fit = X.iloc[idx], y[idx]
    if not USE_EXTRA_ROWS:
        return X_fit, y_fit, np.zeros(len(idx), bool)
    keep = AUG["movie_title"].isin(set(groups.iloc[idx])).values
    return (pd.concat([X_fit, X_aug[keep]], ignore_index=True), np.r_[y_fit, y_aug[keep]],
            np.r_[np.zeros(len(idx), bool), np.ones(keep.sum(), bool)])

def extra_weights(is_extra):
    return np.where(is_extra, EXTRA_WEIGHT, 1.0)

def national_scale(X, power):
    """national real admissions on the row's day / D1-D3 mean (+ 0.05), to the given power; 1 when the power is 0 or the Cinepoint columns are off"""
    if power == 0 or "ret_h" not in X:
        return np.ones(len(X))
    nat = X["ret_h"].fillna(0.7 * X["cut_h"]).fillna(NAT_MEDIAN) + 0.05
    return nat.to_numpy() ** power

NAT_MEDIAN = float(train_df["ret_h"].fillna(0.7 * train_df["cut_h"]).median()) if "ret_h" in train_df else 0.0

def seed_average(make_model, X_tr, y_tr, X_pred, prep=lambda d: d, sample_weight=None):
    preds = [make_model(seed).fit(prep(X_tr), y_tr, sample_weight=sample_weight).predict(prep(X_pred)) for seed in MODEL_SEEDS]
    return np.mean(preds, axis=0)

def cross_validate(name, fit_predict):
    snap = lambda p: np.where(p < ZERO_SNAP.get(name, 0.0), 0.0, p)
    if RUN_CV:
        oof = np.zeros(len(train_df))
        fold_scores = []
        for fold, (tr_idx, va_idx) in enumerate(FOLDS):
            oof[va_idx] = snap(np.clip(fit_predict(*fit_rows(tr_idx), X.iloc[va_idx]), 0, None))
            fold_scores.append(mase_ratio(y[va_idx], oof[va_idx]))
            print(f"{name} fold {fold}: MASE={fold_scores[-1]:.4f}")
        OOF[name], SCORES[name] = oof, fold_scores
        print(f"{name} CV MASE: {mase_ratio(y, oof):.4f}")
    TEST_PRED[name] = snap(np.clip(fit_predict(*fit_rows(np.arange(len(X))), X_test), 0, None))

print(f"features: {len(FEATURES)} | train rows: {len(train_df)} | extra rows: {len(AUG) if USE_EXTRA_ROWS else 0} | test rows: {len(test_df)}")
print(f"naive predict 0       : {mase_ratio(y, 0):.4f}")
print(f"naive predict D1-D3 avg: {mase_ratio(y, 1):.4f}")

incomplete train days: 13 (2025-06-01 .. 2025-06-16); train rows dropped: 1766 of 54671
features: 96 | train rows: 52905 | extra rows: 244229 | test rows: 72611
naive predict 0       : 0.6046
naive predict D1-D3 avg: 0.7356


### LGBM Regressor
LightGBM with an L1 (absolute error) objective and the parameters found by Optuna on 2026-10-07, with 1500 trees instead of 883 (better with the extra rows).

`TARGET_POWER_LGBM = 0.75`: the LGBM predicts tickets / national_ratio^0.75 (national_ratio = the film's real admissions on the row's day / its D1-D3 mean, from Cinepoint) and multiplies it back, with sample weights national_ratio^0.75 so the loss is the same as on the plain target. It extrapolates better to seasons train never saw (Lebaran week has national ratios far above anything in train): on train films held out for having ratios of 1.5 or more, MASE 1.29 -> 1.06, and 0.2963 -> 0.2937 in the usual CV. `TARGET_POWER_KUMO = 0.75` does the same for Kumo. Checked on train films held out for having national ratios of 1.5 or more (Kumo MASE 1.345 at power 0 -> 1.043 at 0.75) and on ordinary rows (0.2920 -> 0.2882); the LGBM is best at 0.75 on both (1.025, CV 0.2855 vs 0.2893).

In [ ]:
LGBM_PARAMS = dict(
    n_estimators=1500,   # 883 in the Optuna run; 1500 is better with the extra rows
    learning_rate=0.0149601,
    num_leaves=87,
    min_child_samples=11,
    subsample=0.641476,
    colsample_bytree=0.48309,
    reg_lambda=1.85904,
    reg_alpha=11.5708,
    subsample_freq=1,
)

def lgbm_fit_predict(X_tr, y_tr, is_extra, X_pred):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    make_model = lambda seed: LGBMRegressor(objective="l1", random_state=seed, verbose=-1, **LGBM_PARAMS)
    s_tr, s_pred = national_scale(X_tr, TARGET_POWER_LGBM), national_scale(X_pred, TARGET_POWER_LGBM)   # weights s_tr keep the loss the same as on the plain target
    pred = seed_average(make_model, X_tr, (y_tr + 0.05) / s_tr, X_pred, sample_weight=extra_weights(is_extra) * s_tr)
    return pred * s_pred - 0.05

if "lgbm" in RUN_MODELS:
    cross_validate("lgbm", lgbm_fit_predict)

### CatBoost Regressor
CatBoost with loss MAE and the Optuna parameters of 2026-10-07 (20 trials, top trials re-scored on 3 seeds). Category columns are passed as text, which CatBoost encodes itself. Default parameters gave 0.3791 and these give 0.3704 averaged over 3 seeds. It is the slowest model: each fit takes 1 to 3 minutes.

In [ ]:
CATBOOST_PARAMS = dict(
    iterations=723,
    learning_rate=0.0329941,
    depth=10,
    l2_leaf_reg=11.9474,
    random_strength=7.56829,
    bagging_temperature=1.78965,
)

def catboost_fit_predict(X_tr, y_tr, is_extra, X_pred):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    to_text = lambda df: df.astype({c: str for c in CAT_COLS})
    make_model = lambda seed: CatBoostRegressor(loss_function="MAE", random_seed=seed, verbose=0, cat_features=CAT_COLS, thread_count=-1, **CATBOOST_PARAMS)
    return seed_average(make_model, X_tr, y_tr, X_pred, prep=to_text, sample_weight=extra_weights(is_extra))

if "catboost" in RUN_MODELS:
    cross_validate("catboost", catboost_fit_predict)

### XGBoost Regressor
XGBoost with an absolute-error objective and the Optuna parameters of 2026-10-07 (60 trials, top trials re-scored on 3 seeds), reading the category columns directly. Default parameters gave 0.4047 and these give 0.3781 averaged over 10 seeds, still the weakest of the three, so it is off by default (add "xgboost" to RUN_MODELS to use it).

In [ ]:
XGB_PARAMS = dict(
    n_estimators=541,
    learning_rate=0.0145094,
    max_depth=10,
    min_child_weight=46.1445,
    subsample=0.514938,
    colsample_bytree=0.431351,
    reg_lambda=7.78404,
    reg_alpha=2.25023,
)

def xgboost_fit_predict(X_tr, y_tr, is_extra, X_pred):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    make_model = lambda seed: XGBRegressor(objective="reg:absoluteerror", enable_categorical=True, tree_method="hist", random_state=seed, **XGB_PARAMS)
    return seed_average(make_model, X_tr, y_tr, X_pred, sample_weight=extra_weights(is_extra))

if "xgboost" in RUN_MODELS:
    cross_validate("xgboost", xgboost_fit_predict)

### Kumo-Tabular
NVIDIA's pretrained tabular foundation model (nvidia/Kumo-Tabular, large). It does not train: a sample of training rows is passed as context and the median of its 999 quantiles is used.
- KUMO_CONTEXT_ROWS / KUMO_EXTRA_ROWS: 8,000 real rows + 2,000 shifted-window rows per run (best clean CV of 10k + 0, 8k + 2k, 6k + 4k, 4k + 6k).
- KUMO_CONTEXT_SEEDS: one run per seed with a different context sample, averaged.

In [ ]:
KUMO_CONTEXT_ROWS = 12000   # real rows in the context (8000 before: on the same validation rows MASE 0.2773 -> 0.2748 at 12000, 0.2736 at 16000 but 3x slower and close to the 8 GB GPU memory)
KUMO_EXTRA_ROWS = 3000    # shifted-window rows in the context (same 4:1 mix as the earlier 8000 + 2000)
KUMO_CONTEXT_SEEDS = [42, 7, 123]
KUMO_ESTIMATORS = 8   # passes of the model per context seed, averaged (4 before: on the same validation rows MASE 0.2882 -> 0.2859 at 8 and 0.2847 at 16, but twice / four times slower)
KUMO_STYPES = {c: "numerical" for c in NUM_FEATURES} | {c: "categorical" for c in CAT_COLS}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
kumo_model = sdm.models.KumoTabular(task="regression", size="large", device=DEVICE).eval() if "kumo" in RUN_MODELS else None

def kumo_fit_predict(X_tr, y_tr, is_extra, X_pred, chunk_rows=2000):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    s_tr, s_pred = national_scale(X_tr, TARGET_POWER_KUMO), national_scale(X_pred, TARGET_POWER_KUMO)
    y_tr = (y_tr + 0.05) / s_tr   # the median of the scaled target times the scale is the median of the original target
    real_idx, extra_idx = np.flatnonzero(~is_extra), np.flatnonzero(is_extra)
    preds = []
    for seed in KUMO_CONTEXT_SEEDS:
        rng = np.random.default_rng(seed)
        ctx = np.r_[rng.choice(real_idx, min(KUMO_CONTEXT_ROWS, len(real_idx)), replace=False),
                    rng.choice(extra_idx, min(KUMO_EXTRA_ROWS, len(extra_idx)), replace=False)].astype(int)
        x_ctx = TableTensor.from_pandas(X_tr.iloc[ctx], KUMO_STYPES, device=DEVICE)
        y_ctx = TableTensor.from_pandas(pd.DataFrame({"y": y_tr[ctx]}), {"y": "numerical"}, device=DEVICE)
        kumo_model.fit(x_ctx, y_ctx, num_estimators=KUMO_ESTIMATORS, generator=torch.Generator(device=DEVICE).manual_seed(seed))
        quantiles = []
        for start in range(0, len(X_pred), chunk_rows):
            x_q = TableTensor.from_pandas(X_pred.iloc[start:start + chunk_rows], KUMO_STYPES, device=DEVICE)
            quantiles.append(kumo_model.predict(x_q).numerical.float().cpu())
        q = torch.cat(quantiles).numpy()
        preds.append(q[:, q.shape[1] // 2])
        torch.cuda.empty_cache()
    return np.mean(preds, axis=0) * s_pred - 0.05

if "kumo" in RUN_MODELS:
    cross_validate("kumo", kumo_fit_predict)

### Neural network (embeddings)
A small MLP with embeddings for the cinema and the categorical columns, trained with the same national-ratio target and weighted L1 loss as the LGBM, 5 seeds averaged. Alone it is weaker than Kumo and LGBM, but it makes different mistakes: in the full pipeline the blend of the three scores about 0.2695 out-of-fold against 0.2736 for Kumo 0.7 / LGBM 0.3. The settings come from an Optuna search (63 trials) on the out-of-fold blend score.

In [ ]:
NN_SEEDS = [1, 2, 3, 4, 5]
NN_EPOCHS = 15
NN_WIDTH, NN_DEPTH, NN_DROPOUT = 512, 4, 0.287
NN_LR, NN_WEIGHT_DECAY, NN_BATCH, NN_EMB_CAP = 7.9e-3, 6.9e-6, 256, 8   # Optuna search, 63 trials (log.md)
TARGET_POWER_NN = 0.75   # same meaning as TARGET_POWER_LGBM
NN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class EmbeddingMLP(torch.nn.Module):
    """MLP on the numeric columns plus embeddings of the categorical columns and the cinema"""
    def __init__(self, n_num, cardinalities, width=NN_WIDTH, depth=NN_DEPTH, dropout=NN_DROPOUT):
        super().__init__()
        self.emb = torch.nn.ModuleList([torch.nn.Embedding(c, min(NN_EMB_CAP, (c + 1) // 2 + 1)) for c in cardinalities])
        dim = n_num + sum(e.embedding_dim for e in self.emb)
        layers = []
        for k in range(depth):
            nxt = width if k == 0 else width // 2
            layers += [torch.nn.Linear(dim, nxt), torch.nn.BatchNorm1d(nxt), torch.nn.SiLU(), torch.nn.Dropout(dropout)]
            dim = nxt
        self.mlp = torch.nn.Sequential(*layers, torch.nn.Linear(dim, 1))

    def forward(self, x_num, x_cat):
        return self.mlp(torch.cat([x_num] + [e(x_cat[:, k]) for k, e in enumerate(self.emb)], dim=1)).squeeze(1)

def nn_fit_predict(X_tr, y_tr, is_extra, X_pred):
    s_tr, s_pred = national_scale(X_tr, TARGET_POWER_NN), national_scale(X_pred, TARGET_POWER_NN)   # same national-ratio target as the other models
    num = [c for c in FEATURES if c not in CAT_COLS]
    ref = X_tr[num].astype(float)
    med, lo, hi = ref.median(), ref.quantile(0.01), ref.quantile(0.99)
    clipped = ref.clip(lo, hi, axis=1)
    mu, sd = clipped.mean(), clipped.std() + 1e-9
    nan_cols = [c for c in num if ref[c].isna().mean() > 0.01]
    cards = [len(X_tr[c].cat.categories) + 2 for c in CAT_COLS] + [len(CINEMA_CODE) + 2]

    def prepare(X):
        v = X[num].astype(float)
        z = ((v.fillna(med).clip(lo, hi, axis=1) - mu) / sd).values
        flags = v[nan_cols].isna().astype(float).values
        codes = np.stack([X[c].cat.codes.values + 1 for c in CAT_COLS] + [X["cinema_code"].values + 1], axis=1)
        return (torch.tensor(np.hstack([z, flags]).astype(np.float32), device=NN_DEVICE), torch.tensor(codes.astype(np.int64), device=NN_DEVICE))

    xn, xc = prepare(X_tr)
    xn_p, xc_p = prepare(X_pred)
    target = torch.tensor(((y_tr + 0.05) / s_tr).astype(np.float32), device=NN_DEVICE)
    weight = torch.tensor((s_tr * extra_weights(is_extra)).astype(np.float32), device=NN_DEVICE)
    preds = []
    for seed in NN_SEEDS:
        torch.manual_seed(seed)
        net = EmbeddingMLP(xn.shape[1], cards).to(NN_DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=NN_LR / 1.5, weight_decay=NN_WEIGHT_DECAY)
        steps = NN_EPOCHS * int(np.ceil(len(xn) / NN_BATCH))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=NN_LR, total_steps=steps)
        for epoch in range(NN_EPOCHS):
            net.train()
            order = torch.randperm(len(xn), device=NN_DEVICE)
            for start in range(0, len(xn), NN_BATCH):
                idx = order[start:start + NN_BATCH]
                if len(idx) < 2:
                    continue
                loss = (weight[idx] * (net(xn[idx], xc[idx]) - target[idx]).abs()).mean()   # weighted L1: the same loss as on the plain target
                opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        net.eval()
        with torch.no_grad():
            preds.append(net(xn_p, xc_p).cpu().numpy())
    return np.mean(preds, axis=0) * s_pred - 0.05

if "nn" in RUN_MODELS:
    cross_validate("nn", nn_fit_predict)

### Model comparison
CV MASE of every model that ran, overall and per fold.

In [ ]:
comparison = pd.DataFrame(
    {name: [mase_ratio(y, OOF[name])] + SCORES[name] for name in OOF},
    index=["CV"] + [f"fold {k}" for k in range(N_FOLDS)],
).T.sort_values("CV") if RUN_CV else None
comparison.round(4) if RUN_CV else print("RUN_CV = False: no CV scores")

RUN_CV = False: no CV scores


### Blend
Weighted average of the models' predictions, then predictions below BLEND_SNAP set to 0. Kumo 0.7 / LGBM 0.3 had the best clean CV; the cell also prints kumo 0.5-0.8.

In [ ]:
BLEND_WEIGHTS = {"kumo": 0.4, "lgbm": 0.3, "nn": 0.3}   # flat optimum on the out-of-fold predictions of the full pipeline (0.2696 vs 0.2736 for kumo 0.7 / lgbm 0.3); Kumo kept a bit higher since the final Kumo run is stronger than its OOF
BLEND_SNAP = 0.1
used = {m: w for m, w in BLEND_WEIGHTS.items() if m in TEST_PRED and w > 0}
total = sum(used.values())
snap_blend = lambda p: np.where(p < BLEND_SNAP, 0.0, p)
test_blend = snap_blend(sum(w * TEST_PRED[m] for m, w in used.items()) / total)
print(f"blend {({m: round(w / total, 3) for m, w in used.items()})}")
if RUN_CV:
    oof_blend = snap_blend(sum(w * OOF[m] for m, w in used.items()) / total)
    print(f"CV MASE: {mase_ratio(y, oof_blend):.4f}")
    if {"kumo", "lgbm"} <= set(OOF):
        for w in (0.5, 0.6, 0.7, 0.8):
            print(f"  kumo {w:.1f} / lgbm {1 - w:.1f}: CV {mase_ratio(y, snap_blend(w * OOF['kumo'] + (1 - w) * OOF['lgbm'])):.4f}")
    display(train_df.assign(err=np.abs(y - oof_blend)).groupby("horizon")["err"].mean().round(4))

# Postprocessing

### Seasonal adjustment
Train starts on Idulfitri 2025, so the model never saw a film whose D1-D3 is at the end of Ramadan and whose D4-D10 is in Lebaran week. For those rows (D1-D3 touches Ramadan and the target date is from 20 March 2026 on) the prediction is multiplied by LEBARAN_FACTOR.
- The earlier factors (lebaran x2.0, libur_sekolah x1.25) were set on the public LB with a weaker model (0.47566 baseline, 0.45395 with main.ipynb and default parameters). With the national-ratio target and the corrected Cinepoint data the model predicts those days differently, so the factors were re-derived from the real national numbers.
- Method: for every film and horizon, our cinemas' predicted ratio (sum over cinemas) is compared with the film's real national ratio on that day times the usual cinema-to-national ratio of the train films; the film's level is the median over its horizons, relative to the level the model has on train films (0.77: the model predicts typical values, below the average). The factor still needed, with big films counting more: Lebaran week 1.65 (1.67 counting every film equally), school-break days outside Lebaran 0.92, Christmas break 0.96, Ramadan days 1.02, other rows 0.89.
- So Lebaran week gets x1.65 (was x2.0) and the school-break factor is dropped (was x1.25: it would now push those rows about 36% too high). Check after a full run: the level of the submission file against the real national numbers should be near 1.00 for these groups.

In [ ]:
RAMADAN_START, LEBARAN_START = ramadan["date"].min(), pd.Timestamp("2026-03-20")
LEBARAN_FACTOR = 1.65   # see the text above

def season_factor(df):
    d13 = [df["d1_date"] + pd.Timedelta(days=k) for k in range(3)]
    d13_ramadan = np.any([(d >= RAMADAN_START) & (d < LEBARAN_START) for d in d13], axis=0)
    lebaran = d13_ramadan & (df["date_show"] >= LEBARAN_START)
    return np.where(lebaran, LEBARAN_FACTOR, 1.0)

test_df["season_factor"] = season_factor(test_df)
final_ratio = test_blend * test_df["season_factor"].values   # ratio after the season factor
test_df["total_ticket"] = final_ratio * test_df["scale"]
test_df["season_factor"].value_counts()

season_factor
1.00    68013
1.65     4598
Name: count, dtype: int64

# Submission

In [ ]:
def write_submission(ratio, path):
    out = test_df[["id"]].astype({"id": int}).assign(total_ticket=ratio * test_df["scale"].values)
    sub = sample_sub[["id"]].merge(out, on="id", how="left")
    assert len(sub) == len(sample_sub) and sub["total_ticket"].notna().all()
    sub.to_csv(path, index=False)
    return sub

submission = write_submission(final_ratio, "submission_cine.csv")
submission.describe()

,id,total_ticket
count,72611.000000,72611.000000
mean,36306.000000,183.180109
std,20961.134535,667.505097
min,1.000000,0.000000
25%,18153.500000,0.000000
50%,36306.000000,25.983429
75%,54458.500000,139.539766
max,72611.000000,38164.390207
